## Objective

The objective of this notebook is to prepare the final production-ready fraud detection model by combining the trained Random Forest model with the selected probability threshold. The notebook also demonstrates how the model can be used to make predictions on new transactions and summarizes the complete machine learning workflow.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

import joblib

In [2]:
rf_smote_model = joblib.load(
    "../models/random_forest_smote.pkl"
)

In [3]:
best_threshold = joblib.load(
    "../models/best_threshold.pkl"
)

print(best_threshold)

0.6000000000000002


In [6]:
# Load processed test datasets
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv")

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


X_test shape: (56746, 30)
y_test shape: (56746, 1)


In [4]:
def predict_transaction(model, threshold, transaction):

    probability = model.predict_proba(transaction)[0][1]

    prediction = (
        "Fraud"
        if probability >= threshold
        else "Legitimate"
    )

    return prediction, probability

In [7]:
sample_transaction = X_test.iloc[[10]]

prediction, probability = predict_transaction(
    rf_smote_model,
    best_threshold,
    sample_transaction
)

print("Prediction :", prediction)
print("Fraud Probability :", probability)

Prediction : Legitimate
Fraud Probability : 0.0


In [9]:
sample_transaction = X_test.iloc[[100]]

prediction, probability = predict_transaction(
    rf_smote_model,
    best_threshold,
    sample_transaction
)

print("Prediction :", prediction)
print("Fraud Probability :", probability)

Prediction : Legitimate
Fraud Probability : 0.02


In [10]:
fraud_detection_pipeline = {
    "Model": rf_smote_model,
    "Threshold": best_threshold
}

In [11]:
joblib.dump(
    fraud_detection_pipeline,
    "../models/fraud_detection_pipeline.pkl"
)

['../models/fraud_detection_pipeline.pkl']

In [12]:
summary = pd.DataFrame({

    "Stage":[
        "Best Model",
        "Sampling Technique",
        "ROC-AUC",
        "PR-AUC",
        "Best Threshold"
    ],

    "Result":[
        "Random Forest",
        "SMOTE",
        0.9591,
        0.8078,
        best_threshold
    ]

})

summary

,Stage,Result
0,Best Model,Random Forest
1,Sampling Technique,SMOTE
2,ROC-AUC,0.9591
3,PR-AUC,0.8078
4,Best Threshold,0.6


In [13]:
summary.to_csv(
    "../reports/project_summary.csv",
    index=False
)

## Business Recommendations

### Deploy the Random Forest model trained using SMOTE.

### Use the optimized probability threshold instead of the default threshold of 0.50.

### Monitor Precision and Recall regularly because fraud patterns evolve over time.

### Retrain the model periodically using newly collected transaction data.

### Use the model as a decision-support tool to flag suspicious transactions for review rather than replacing human investigation.

## Future Improvements

- Perform hyperparameter tuning using GridSearchCV or RandomizedSearchCV.
- Compare additional imbalance techniques such as ADASYN or SMOTE-Tomek.
- Evaluate ensemble methods like LightGBM or CatBoost.
- Use SHAP values for local and global model explainability.
- Develop a real-time fraud detection API using FastAPI.
- Build an interactive dashboard using Streamlit for fraud monitoring.
- Monitor model drift and schedule periodic retraining.

## Conclusion

This project developed a complete machine learning pipeline for detecting fraudulent credit card transactions. After comparing multiple classification algorithms, Random Forest was selected as the strongest baseline model. Applying SMOTE improved the model's ability to identify fraudulent transactions, resulting in an ROC-AUC of 0.9591 and a PR-AUC of 0.8078. Further optimisation through threshold tuning produced a decision rule better aligned with fraud detection objectives than the default threshold. The final deliverable is a production-ready fraud detection pipeline consisting of the trained model and its optimized threshold.